In [ ]:
import pandas as pd
import numpy as np
import csv
import sys
import os
from importlib import reload
sys.path.append("/u/<username>/local/pyenvs/pymeasurements")
import plot_utils as pu
reload(pu)
%matplotlib inline

## Load campus trace RTT and geolocation-related data

In [ ]:
ext_rtt_path = "data/campus_trace_ext_rtt.csv"
flow_map_path = "data/campus_trace_flow_map.csv"
geoloc_path = "data/campus_trace_geolocation_map.csv"
ext_ip_map_path = "data/campus_trace_external_ip_map.csv"
int_ip_map_path = "data/campus_trace_internal_ip_map.csv"

In [ ]:
df_geo = pd.read_csv(geoloc_path, dtype={"Latitude": str, "Longitude": str})
print(df_geo.shape[0])
df_geo.head(n=1)

In [ ]:
df_ext_ip_map = pd.read_csv(ext_ip_map_path)
df_ext_ip_map['External_Prefix'] = df_ext_ip_map['External_IP'].apply(lambda x: ".".join(x.split(".")[:-1]+["0"]))
print(df_ext_ip_map.shape[0])
df_ext_ip_map.head(n=1)

In [ ]:
df_flow_map = pd.read_csv(flow_map_path)
df_flow_map = df_flow_map.rename(columns={'Destination_IP': 'External_IP_ID'})
print(df_flow_map.shape[0])
df_flow_map.head(n=1)

In [ ]:
df_ext_rtt = pd.read_csv(ext_rtt_path)
print(df_ext_rtt.shape[0])
df_ext_rtt.head(n=1)

In [ ]:
df_ext_rtt_count = df_ext_rtt.groupby('Flow_ID').size().reset_index(name='RTT_Samples')
print(df_ext_rtt_count.shape[0])
df_ext_rtt_count.head(n=1)

## Merge external IPs and geolocation data

In [ ]:
df_merged_extip_geo = pd.merge(df_ext_ip_map, df_geo, on='External_IP_ID', how='left')
print(df_merged_extip_geo.shape[0])
df_merged_extip_geo.head(n=1)

In [ ]:
df_merged_extip_geo_usa = df_merged_extip_geo[df_merged_extip_geo['Country']=='United States'][['External_IP', 'External_Prefix']]
print(df_merged_extip_geo_usa.shape[0])
df_merged_extip_geo_usa.head(n=1)

In [ ]:
count_ip = df_merged_extip_geo_usa['External_IP'].nunique()
count_pr = df_merged_extip_geo_usa['External_Prefix'].nunique()
print(f"Count of individual external IPs: {count_ip}")
print(f"Count of external IP prefixes: {count_pr} ({round(count_pr*100.0/count_ip, 2)}%)")
prefix_counts = df_merged_extip_geo_usa.groupby('External_Prefix').size().tolist()
pu.plot_cdf(prefix_counts, xlabel="#IPs per external /24 prefix", xscale="log", title="",
            trans_bg=False, all_text_color="black")

## Merge flow info and external info and geolocation data

In [ ]:
df_merged_flow_extip_geo = pd.merge(df_flow_map, df_merged_extip_geo, on='External_IP_ID', how='left')
print(df_merged_flow_extip_geo.shape[0])
df_merged_flow_extip_geo.head(n=1)

In [ ]:
df_merged_flow_extip_geo_usa = df_merged_flow_extip_geo[df_merged_flow_extip_geo['Country']=='United States'][
    ['Flow_ID', 'Source_IP', 'External_IP', 'External_Prefix', 'Source_Port', 'Destination_Port']]
print(df_merged_flow_extip_geo_usa.shape[0])
df_merged_flow_extip_geo_usa.head(n=1)

In [ ]:
df_merged_flow_extip_geo_usa_pucli = df_merged_flow_extip_geo_usa.loc[
    (df_merged_flow_extip_geo_usa['Source_Port'] > 1023) & (df_merged_flow_extip_geo_usa['Destination_Port'] < 1024)]
print(df_merged_flow_extip_geo_usa_pucli.shape[0])
print(f"Percentage of flows with a PU client: {round(df_merged_flow_extip_geo_usa_pucli.shape[0]*100.0/df_merged_flow_extip_geo_usa.shape[0], 2)}")

In [ ]:
df_merged_flow_extip_geo_usa_pusrv = df_merged_flow_extip_geo_usa.loc[
    (df_merged_flow_extip_geo_usa['Source_Port'] < 1024) & (df_merged_flow_extip_geo_usa['Destination_Port'] > 1023)]
print(df_merged_flow_extip_geo_usa_pusrv.shape[0])
print(f"Percentage of flows with a PU server: {round(df_merged_flow_extip_geo_usa_pusrv.shape[0]*100.0/df_merged_flow_extip_geo_usa.shape[0], 2)}")

In [ ]:
flow_freq_all = df_merged_flow_extip_geo_usa.groupby('External_Prefix').size().tolist()
flow_freq_cli = df_merged_flow_extip_geo_usa_pucli.groupby('External_Prefix').size().tolist()
flow_freq_srv = df_merged_flow_extip_geo_usa_pusrv.groupby('External_Prefix').size().tolist()
pu.plot_cdfs([flow_freq_all, flow_freq_cli, flow_freq_srv], curvelabels=['All', 'Client', 'Server'], xlabel="#Flows per external /24 prefix", xscale="log", title="",
            trans_bg=False, all_text_color="black")

## Merge RTT data with flow, external IP, geo info

In [ ]:
df_merged_flow_extip_geo_rtt = pd.merge(df_merged_flow_extip_geo, df_ext_rtt_count, on='Flow_ID', how='left')
print(df_merged_flow_extip_geo_rtt.shape[0])
df_merged_flow_extip_geo_rtt.head(n=1)

In [ ]:
df_merged_flow_extip_geo_rtt_usa = df_merged_flow_extip_geo_rtt[df_merged_flow_extip_geo_rtt['Country']=='United States'][
    ['Flow_ID', 'Source_IP', 'External_IP', 'External_Prefix', 'Source_Port', 'Destination_Port', 'RTT_Samples']]
print(df_merged_flow_extip_geo_rtt_usa.shape[0])
df_merged_flow_extip_geo_rtt_usa.head(n=1)

In [ ]:
df_merged_flow_extip_geo_rtt_usa_pucli = df_merged_flow_extip_geo_rtt_usa.loc[
    (df_merged_flow_extip_geo_rtt_usa['Source_Port'] > 1023) & (df_merged_flow_extip_geo_rtt_usa['Destination_Port'] < 1024)]
print(df_merged_flow_extip_geo_rtt_usa_pucli.shape[0])
print(f"Percentage of flows with a PU client: {round(df_merged_flow_extip_geo_rtt_usa_pucli.shape[0]*100.0/df_merged_flow_extip_geo_rtt_usa.shape[0], 2)}")

In [ ]:
df_merged_flow_extip_geo_rtt_usa_pusrv = df_merged_flow_extip_geo_rtt_usa.loc[
    (df_merged_flow_extip_geo_rtt_usa['Source_Port'] < 1024) & (df_merged_flow_extip_geo_rtt_usa['Destination_Port'] > 1023)]
print(df_merged_flow_extip_geo_rtt_usa_pusrv.shape[0])
print(f"Percentage of flows with a PU server: {round(df_merged_flow_extip_geo_rtt_usa_pusrv.shape[0]*100.0/df_merged_flow_extip_geo_rtt_usa.shape[0], 2)}")

In [ ]:
rtt_freq_all = df_merged_flow_extip_geo_rtt_usa.groupby('External_Prefix')['RTT_Samples'].sum().tolist()
rtt_freq_cli = df_merged_flow_extip_geo_rtt_usa_pucli.groupby('External_Prefix')['RTT_Samples'].sum().tolist()
rtt_freq_srv = df_merged_flow_extip_geo_rtt_usa_pusrv.groupby('External_Prefix')['RTT_Samples'].sum().tolist()
pu.plot_cdfs([rtt_freq_all, rtt_freq_cli, rtt_freq_srv], curvelabels=['All', 'Client', 'Server'], xlabel="#RTTs per external /24 prefix", xscale="log", title="",
            trans_bg=False, all_text_color="black")